# MAS-RAG Universal Context Engine (Oracle UI Edition)

Copyright 2025-2026, Denis Rothman

## **Overview**
This notebook implements an **Agentic RAG Architecture** powered by **Oracle 23ai**. It demonstrates the "Converged Database" concept by seamlessly switching between **Vector Search** (for unstructured knowledge) and **Relational SQL** (for structured HR data) within a single execution engine.

## **Key Features**
* **Interactive Control Deck:** A widget-based UI that allows users to select preset scenarios or input custom natural language goals.
* **Trace Dashboard:** A high-contrast HTML visualization that tracks agent execution steps, token usage (input/output/saved), and latency in real-time.
* **Hybrid Agents:**
    * **`OracleRecruiter`:** Performs hybrid retrieval by combining SQL constraints (e.g., salary, experience) with vector similarity searches.
    * **`OracleArchivist`:** Retrieves unstructured text chunks from the knowledge store using semantic search.
* **Universal Architecture:** Utilizes a centralized Planner-Executor framework (`engine.py`) that dynamically generates execution plans based on available capabilities.



# 1.Installation & Setup

In [ ]:
#@title Retrieving GitHub Private token (this cell will be deleted when the repository is public)
import os
from openai import OpenAI
from google.colab import userdata

# Load the API key from Colab secrets, set the env var, then init the client
try:
    pt = userdata.get("GITHUB_TOKEN")
    pt=pt.strip()
    if not pt:
        raise userdata.SecretNotFoundError("GITHUB_TOKEN not found.")

    print("GITHUB_TOKENAPI private token loaded successfully.")

except userdata.SecretNotFoundError:
    print('Secret "GITHUB_TOKEN" not found.')
    print('Please activate or add your GITHUB_TOKEN private token to the Colab Secrets Manager.')
except Exception as e:
    print(f"An error occurred while loading the GITHUB_TOKEN: {e}")

In [ ]:
# Install Oracle 23ai and Frozen LLM dependencies
# We lock versions to prevent internal Pydantic/OpenAI parsing conflicts
!pip install --upgrade --force-reinstall \
    openai==2.26.0 \
    pydantic==2.11.9 \
    tiktoken==0.12.0 \
    tqdm==4.67.1 \
    tenacity==9.1.2 \
    oracledb==3.4.1 \
    requests==2.32.4 \
    cryptography==43.0.3 \
    pyOpenSSL==24.2.1 \
    --quiet

print("✅ Oracle 23ai and Frozen LLM dependencies installed.")

In [ ]:
import os
import sys
from google.colab import drive, userdata
from openai import OpenAI

# Mount Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

In [ ]:
# Initialize Oracle Client

try:
    # Explicitly check for the mandatory OpenAI key
    openai_key = userdata.get("API_KEY")
    if not openai_key:
        raise ValueError("OpenAI API_KEY not found in Secrets.")

    os.environ["OPENAI_API_KEY"] = openai_key
    client = OpenAI()
    print("✅ OpenAI initialized (Mandatory).")

except Exception as e:
    print(f"❌ Critical Error: OpenAI initialization failed. {e}")
    # We raise the error here because OpenAI is required for the Planner to work at all
    raise e

In [ ]:
#@title Retrieving engine library
import time

!curl -L -H "Authorization: token {pt}" -H "Accept: application/vnd.github.v3.raw" "https://api.github.com/repos/Denis2054/RAG-Driven-Generative-AI-2nd-Edition/contents/commons/engine/engine.zip" -o engine.zip

time.sleep(10)
!unzip -o engine.zip -d /content

In [ ]:
# 3. Import Local Modules (Uploaded Files)
try:
    from engine import context_engine
    from registry import AGENT_TOOLKIT
    from oracle_lib import OracleManager
    print("✅ Universal Engine Modules imported.")
except ImportError as e:
    print(f"❌ Module Import Failed: {e}\nUpload engine.py, registry.py, agents.py, helpers.py, oracle_lib.py, agent_archivist.py, agent_recruiter.py")

# 2.Initialize the Universal Engine

In [ ]:
# Initialize Oracle Connection (Enterprise Layer)
try:
    OracleManager.initialize()
    print("✅ Oracle 23ai Connection established.")
except Exception as e:
    print(f"⚠️ Oracle Connection Failed: {e}\n(Ignore if you only want to run Pinecone scenarios)")

The following cell contains `ensure_oracle_connection()`, a utility function that can be called from anywhere in the application. This is particularly useful if your network connection experiences micro-disconnections.

Usage: Insert `ensure_oracle_connection()` anywhere necessary. The function checks the status of your Oracle connection and ensures it is active before proceeding.


In [ ]:
def ensure_oracle_connection():
    # Tell Python we want to modify the variable that lives outside this function
    global oracle_is_connected

    # Check if the variable exists safely; if not, default to False
    if 'oracle_is_connected' not in globals():
        oracle_is_connected = False

    if oracle_is_connected:
        print("✅ Oracle connection is already active.")
    else:
        print("⚠️ Oracle NOT connected. Attempting initialization...")
        try:
            OracleManager.initialize()
            oracle_is_connected = True  # This now updates the global variable directly!
            print("✅ Oracle 23ai Connection established.")
        except Exception as e:
            print(f"⚠️ Connection Failed: {e}")
            oracle_is_connected = False

ensure_oracle_connection()

In [ ]:
# We define a wrapper function that pre-configures the engine with our clients and models.
# This allows us to just pass the 'goal' in the scenario steps below.
def run_universal_engine(goal):
    return context_engine(
        goal=goal,
        client=client,
        # [FIX] Removed Pinecone Client (pc), Index, and Namespaces
        generation_model="gpt-5.1",
        embedding_model="text-embedding-3-small"
    )

# I. Inititalization UCE

## Render and Trace Dashboard

In [ ]:
import json
import html
import markdown
from IPython.display import display, HTML

def render_trace_dashboard(trace):
    """
    Generates a high-contrast, high-readability HTML dashboard for the Context Engine Trace.
    Includes explicit token metrics and deep charcoal text for maximum clarity.
    """
    # Define CSS with hard-coded high-contrast colors and heavy font weights
    css = """
    <style>
        .dashboard-container {
            font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif;
            background-color: #ffffff;
            border: 3px solid #cbd5e0;
            border-radius: 12px;
            padding: 30px;
            max-width: 100%;
            margin-top: 25px;
            color: #1a202c;
        }
        .header-section {
            border-bottom: 3px solid #2d3748;
            padding-bottom: 20px;
            margin-bottom: 25px;
            display: flex;
            justify-content: space-between;
            align-items: center;
        }
        .header-title { margin: 0; font-size: 1.8rem; color: #1a202c; font-weight: 900; }
        .header-goal { margin: 10px 0 0 0; color: #2d3748; font-size: 1.2rem; font-style: italic; font-weight: 600;}

        .status-badge {
            padding: 10px 20px;
            border-radius: 8px;
            font-weight: 900;
            font-size: 1rem;
            color: white;
            text-transform: uppercase;
        }
        .status-success { background-color: #22543d; }
        .status-failure { background-color: #742a2a; }

        .metrics-summary {
            margin-top: 10px;
            font-size: 1.2rem;
            font-weight: 900;
            color: #1a202c;
            background: #edf2f7;
            padding: 5px 12px;
            border-radius: 6px;
        }

        /* Metrics Display within Steps */
        .metrics-bar {
            display: flex;
            gap: 12px;
            margin-top: 12px;
            flex-wrap: wrap;
        }
        .metric-pill {
            background-color: #ebf4ff;
            color: #1a365d; /* High contrast dark blue */
            padding: 6px 14px;
            border-radius: 8px;
            border: 2px solid #2b6cb0;
            font-size: 0.95rem;
            font-weight: 900;
        }
        .metric-saved {
            background-color: #f0fff4;
            color: #1c4532;
            border-color: #2f855a;
        }

        /* Step Cards */
        .step-card {
            background-color: #ffffff;
            border: 2px solid #2d3748;
            border-radius: 12px;
            margin-bottom: 25px;
            overflow: hidden;
            box-shadow: 0 4px 6px rgba(0,0,0,0.05);
        }
        summary.step-header {
            padding: 20px;
            background-color: #f8fafc;
            cursor: pointer;
            list-style: none;
            display: flex;
            align-items: center;
            justify-content: space-between;
            border-bottom: 2px solid #e2e8f0;
        }
        .agent-badge {
            background-color: #1a202c;
            color: #ffffff;
            padding: 5px 14px;
            border-radius: 6px;
            font-size: 0.85rem;
            font-weight: 900;
            text-transform: uppercase;
            margin-left: 15px;
        }

        .step-content { padding: 25px; background-color: #ffffff; }

        .data-label {
            font-size: 1rem;
            text-transform: uppercase;
            color: #1a202c;
            font-weight: 900;
            margin-bottom: 12px;
            display: block;
            border-left: 4px solid #1a202c;
            padding-left: 10px;
        }

        /* THE CONTENT AREA - High Visibility for Headings */
        .rendered-content {
            background-color: #ffffff;
            border: 2px solid #e2e8f0;
            border-left: 8px solid #2b6cb0;
            padding: 20px;
            color: #1a202c !important;
            line-height: 1.7;
            font-size: 1.1rem;
            font-weight: 500;
        }
        .rendered-content h1, .rendered-content h2, .rendered-content h3,
        .rendered-content h4, .rendered-content h5, .rendered-content h6 {
            color: #1a202c !important;
            font-weight: 900 !important;
            margin-top: 1.5em;
            margin-bottom: 0.5em;
        }
        .rendered-content strong, .rendered-content b {
            font-weight: 900;
            color: #000000;
        }

        .json-box {
            background-color: #1a202c;
            color: #f7fafc;
            padding: 20px;
            border-radius: 10px;
            font-family: "SFMono-Regular", Consolas, monospace;
            font-size: 0.95rem;
            overflow-x: auto;
        }

        .final-output-card {
            border: 5px solid #22543d;
            background-color: #f0fff4;
            border-radius: 12px;
            padding: 30px;
            margin-top: 50px;
            color: #1a202c;
        }
    </style>
    """

    status_class = "status-success" if trace.status == "Success" else "status-failure"

    dashboard_html = f"""
    {css}
    <div class="dashboard-container">
        <div class="header-section">
            <div>
                <h1 class="header-title">Context Engine Trace</h1>
                <p class="header-goal">"{html.escape(trace.goal)}"</p>
            </div>
            <div style="text-align: right;">
                <span class="status-badge {status_class}">{trace.status}</span>
                <div class="metrics-summary">
                    ⏱️ TIME: {trace.duration:.2f}s
                </div>
            </div>
        </div>

        <div class="steps-container">
            <h2 style="color: #1a202c; margin-bottom: 25px; font-size: 1.4rem; font-weight: 900; text-transform: uppercase;">Execution Workflow</h2>
    """

    for step in trace.steps:
        # Prepare inputs and outputs
        try:
            resolved_ctx = json.dumps(step['resolved_context'], indent=2)
        except:
            resolved_ctx = str(step.get('resolved_context', 'N/A'))

        output_raw = step.get('output', 'N/A')
        rendered_html = ""

        def render_md(text):
            return markdown.markdown(text) if text else "No content recorded."

        if isinstance(output_raw, dict):
            content_found = False
            for key in ['summary', 'answer_with_sources', 'answer', 'output', 'content']:
                if key in output_raw and isinstance(output_raw[key], str):
                    rendered_html = f'<div class="rendered-content">{render_md(output_raw[key])}</div>'
                    content_found = True
                    break
            if not content_found:
                rendered_html = f'<div class="json-box">{html.escape(json.dumps(output_raw, indent=2))}</div>'
        else:
            rendered_html = f'<div class="rendered-content">{render_md(str(output_raw))}</div>'

        # Metrics Extraction
        t_in = step.get('tokens_in', '??')
        t_out = step.get('tokens_out', '??')
        t_saved = step.get('tokens_saved', 0)

        metrics_html = f"""
        <div class="metrics-bar">
            <span class="metric-pill">📥 IN: {t_in}</span>
            <span class="metric-pill">📤 OUT: {t_out}</span>
        """
        if t_saved > 0:
            metrics_html += f'<span class="metric-pill metric-saved">📉 SAVED: {t_saved}</span>'
        metrics_html += "</div>"

        # Build Step Card
        step_html = f"""
            <details class="step-card" open>
                <summary class="step-header">
                    <div style="display:flex; flex-direction:column; align-items:flex-start;">
                        <div>
                            <span style="font-weight:900; font-size: 1.3rem; color: #1a202c;">STEP {step['step']}</span>
                            <span class="agent-badge">{step['agent']}</span>
                        </div>
                        {metrics_html}
                    </div>
                    <span style="font-weight: 900; color: #ffffff; background: #1a202c; padding: 6px 14px; border-radius: 6px; font-size: 0.8rem;">OPEN LOGS</span>
                </summary>
                <div class="step-content">
                    <div style="margin-bottom:30px;">
                        <span class="data-label">Input Context (State)</span>
                        <details><summary style="font-size:0.9rem; font-weight:900; color: #2b6cb0; cursor:pointer; margin-bottom:10px;">▶ View Resolved Source Data</summary>
                        <div class="json-box">{html.escape(resolved_ctx)}</div></details>
                    </div>
                    <div>
                        <span class="data-label">Agent Output</span>
                        {rendered_html}
                    </div>
                </div>
            </details>
        """
        dashboard_html += step_html

    # Final Result Section
    if trace.final_output:
        final_content = trace.final_output
        if isinstance(final_content, dict):
            final_content = final_content.get('summary', final_content.get('content', str(final_content)))

        dashboard_html += f"""
        <div class="final-output-card">
            <div style="color:#1c4532; font-size: 1.5rem; font-weight:1000; margin-bottom:20px; text-transform: uppercase; letter-spacing: 2px; border-bottom: 3px solid #22543d; padding-bottom: 12px;">Final Orchestration Result</div>
            <div style="font-size: 1.25rem; font-weight: 700; line-height: 1.8;">{markdown.markdown(str(final_content))}</div>
        </div>
        """

    dashboard_html += "</div>"
    display(HTML(dashboard_html))

## Engine Room

In [ ]:
# === ENGINE ROOM: The Main Execution Function (Oracle Edition) ===

import logging
import pprint
import json
from IPython.display import display, Markdown

# Import the Oracle Manager to ensure we can connect
from oracle_lib import OracleManager

def execute_and_display(goal, config, client, moderation_active): # <--- REMOVED 'pc'
    """
    Runs the context engine with HTML dashboard visualization.
    """
    # 0. Initialize Oracle Connection (if not already open)
    try:
        # Update these paths to where you stored your wallet/credentials in Colab
        OracleManager.initialize(
            creds_path="/content/drive/MyDrive/files/oracle/credentials.txt",
            wallet_path="/content/drive/MyDrive/files/oracle"
        )
    except Exception as e:
        print(f"🛑 Oracle Connection Failed: {e}")
        return

    # --- PRE-FLIGHT MODERATION CHECK (on user input) ---
    if moderation_active:
        print("--- [Safety Guardrail] Performing Pre-Flight Moderation Check on Goal ---")
        moderation_report = helpers.helper_moderate_content(text_to_moderate=goal, client=client)

        if moderation_report["flagged"]:
            print("\n🛑 Goal failed pre-flight moderation. Execution halted.")
            pprint.pprint(moderation_report)
            return

    logging.info(f"******** Starting Engine for Goal: '{{goal}}' *********\n")

    # 1. Run the Context Engine
    # [FIX] We map the config keys to the specific arguments engine.py expects
    result, trace = context_engine(
        goal,
        client=client,
        generation_model=config["generation_model"],
        embedding_model=config["embedding_model"]
        # Note: 'pc', 'index_name', and namespaces are NO LONGER passed.
    )

    # --- POST-FLIGHT MODERATION CHECK (on AI output) ---
    if result and moderation_active:
        text_to_check = str(result)
        if isinstance(result, (dict, list)):
            text_to_check = json.dumps(result)

        moderation_report = helpers.helper_moderate_content(text_to_moderate=text_to_check, client=client)

        if moderation_report["flagged"]:
            print("\n🛑 Generated output failed post-flight moderation and will be redacted.")
            result = "[Content flagged as potentially harmful by moderation policy and has been redacted.]"
            trace.final_output = result

    # 2. Render the HTML Dashboard
    if trace:
        render_trace_dashboard(trace)
    else:
        print("Engine failed to initialize trace.")

In [ ]:
# =============================================================================
# 🛠️ ROBUST PLANNER PATCH
# =============================================================================
# Run this cell to overwrite the default planner with a strict-schema version.
# This fixes the "NoneType object has no attribute 'get'" error.

import engine
import json
import logging
from helpers import call_llm_robust

def planner_robust_patch(goal, capabilities, client, generation_model):
    """
    A patched version of the Planner that enforces strict JSON schema for inputs.
    """
    logging.info("[Planner-Patch] Activated. Generating strict schema plan...")

    # UPGRADE: Explicitly showing the required schema in the prompt
    system_prompt = f"""
You are the strategic core of the Context Engine. Analyze the user's high-level GOAL and create a step-by-step EXECUTION PLAN.

AVAILABLE CAPABILITIES
---
{capabilities}
---
END CAPABILITIES

INSTRUCTIONS:
1. The output MUST be a single JSON object with a "plan" key containing a list of step objects.
2. CRITICAL: Every step object MUST strictly follow this schema:
   {{
      "step": <integer>,
      "agent": "<Agent Name>",
      "input": {{
          "<input_key>": "<input_value>"
      }}
   }}
   (You MUST wrap the agent arguments inside the "input" object. Do not place them at the root of the step object.)

3. Use Context Chaining: format "$$STEP_N_OUTPUT$$" for values requiring previous outputs.
"""
    try:
        plan_json_string = call_llm_robust(
            system_prompt,
            goal,
            client=client,
            generation_model=generation_model,
            json_mode=True
        )
        plan_data = json.loads(plan_json_string)

        # Extra safety: Check if 'plan' key exists, if not, assume the list is the plan
        if "plan" not in plan_data and isinstance(plan_data, list):
             return plan_data
        return plan_data["plan"]

    except Exception as e:
        logging.error(f"Planner failed to generate a valid plan. Error: {e}")
        raise e

# Apply the patch to the imported engine module
engine.planner = planner_robust_patch
print("✅ Engine successfully patched with Robust Planner.")

Business rules

`business_rules` contains a list of keyworks the HR user must enter in a prompt to be Oracle HR compliant. If a keyword is missing, the prompt will be rejected.

if `business_rules` is empty, all prompts will be accepted.

This list can also contain safety filters such as a list of confidential information the user cannot request such as "gender", e.g., if this is a company policy for recruitment.



In [ ]:
# Enterprise Business Rules Configuration
business_rules = {
    # Words the prompt MUST contain to be useful/compliant
    "required": ["experience","salary"],

    # Words the prompt MUST NOT contain (Safety Filters)
    "forbidden": ["gender","female","male", "religion", "politics"]
}

# NOTE: If you set business_rules = {}, the validation checks below will simply pass.

In [ ]:
def validate_prompt(prompt, rules):
    """
    Scans the prompt against business rules.
    Returns: (status: Boolean, message: String)
    """
    # Rule 1: If rules are empty, accept everything
    if not rules:
        return True, "✅ Prompt accepted (No rules active)."

    # Convert prompt to lowercase for case-insensitive matching
    prompt_lower = prompt.lower()

    # Rule 2: Safety Check (Forbidden Words)
    # logic: if ANY forbidden word is IN the prompt -> FAIL
    forbidden_list = rules.get("forbidden", [])
    for word in forbidden_list:
        if word.lower() in prompt_lower:
            return False, f"⛔ Rejected: The prompt contains restricted content: '{word}'"

    # Rule 3: Compliance Check (Required Words)
    # logic: if ANY required word is NOT IN the prompt -> FAIL
    required_list = rules.get("required", [])
    missing_words = []

    for word in required_list:
        if word.lower() not in prompt_lower:
            missing_words.append(word)

    if missing_words:
        return False, f"⚠️ Rejected: Missing required business keywords: {missing_words}"

    # If we survive all checks:
    return True, "✅ Prompt compliant."

#III.CONTROL DECKS

=== CONTROL DECK: Define Goal and Run Engine ===
This is the main interactive cell.
1. Change the 'goal' variable to your desired task.
2. Run this cell.


In [ ]:
def ensure_oracle_connection():
    # Tell Python we want to modify the variable that lives outside this function
    global oracle_is_connected

    # Check if the variable exists safely; if not, default to False
    if 'oracle_is_connected' not in globals():
        oracle_is_connected = False

    if oracle_is_connected:
        print("✅ Oracle connection is already active.")
    else:
        print("⚠️ Oracle NOT connected. Attempting initialization...")
        try:
            OracleManager.initialize()
            oracle_is_connected = True  # This now updates the global variable directly!
            print("✅ Oracle 23ai Connection established.")
        except Exception as e:
            print(f"⚠️ Connection Failed: {e}")
            oracle_is_connected = False

ensure_oracle_connection()

In [ ]:
# Define the missing configuration dictionary
config = {
    "generation_model": "gpt-5.1",             # Or "gpt-3.5-turbo"
    "embedding_model": "text-embedding-3-small"
}

# Ensure the OpenAI client is active (from previous cells)
# If 'client' is also missing, uncomment the line below:
# from openai import OpenAI; client = OpenAI()

## Control deck instructions

Rerun the control deck cell between each scenario for this educational notebook.

Technical Note on State Persistence explanation:  This interactive Control Deck is designed for educational transparency. Because the engine resolves execution steps *in-place* to demonstrate real-time variable injection, the internal Plan object in memory may retain resolved strings from a previous scenario (a side-effect of object mutability). Rerunning the cell ensures a *clean slate* by re-initializing the UI state and clearing the kernel memory, guaranteeing that each scenario starts with a fresh, unmutated set of agent instructions.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# 1. Define Presets
presets = {
    "--- ✨ Select to Clear / Enter Custom Goal ---": "",
    "1. Recruitment: Senior Python Dev (Hybrid)": "Find an Experienced Python developer with experience and 250000 max salary and write a job interview email",
    "2. Recruitment: Java Banking Dev (Hybrid)": "Find a Backend Java Developer with banking experience and a max salary of 150000 and summarize the candidates",
    "3. Recruitment: Vague Request (Edge Case)": "Find a developer",
}

# 2. Create Widgets

# Dropdown
goal_dropdown = widgets.Dropdown(
    options=presets.items(),
    value="",
    description='<b>Load Goal:</b>',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='98%')
)

# Text Area
goal_input = widgets.Textarea(
    value="",
    placeholder='Type your CUSTOM GOAL here, or select a preset from the menu above to auto-fill this box...',
    description='<b>Goal Input:</b>',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='98%', height='100px')
)

# Run Button
run_button = widgets.Button(
    description='Run Context Engine',
    button_style='success', # Green
    layout=widgets.Layout(width='30%'),
    icon='play'
)

# Output Area
output_area = widgets.Output()

# 3. Interaction Logic

def on_dropdown_change(change):
    """Update text area when dropdown selection changes."""
    goal_input.value = change['new']

def on_run_click(b):
    """Executes the engine with the current text area content."""
    user_goal = goal_input.value.strip()

    # FIX: Default to True since the checkbox UI is commented out
    is_moderated = False

    # 1. Update UI to indicate processing
    run_button.description = "Validating..."
    run_button.disabled = True

    with output_area:
        clear_output() # Clear previous run

        # --- STEP 0: Empty Check ---
        if not user_goal:
            print("⚠️ Please enter a goal or select a preset.")
            run_button.description = "Run Context Engine"
            run_button.disabled = False
            return

        # --- STEP 1: BUSINESS LOGIC CHECK (New Feature) ---
        # We call your validation function here before the engine starts
        is_valid, rule_message = validate_prompt(user_goal, business_rules)

        if not is_valid:
            # If rejected, print the specific reason (e.g. missing salary) and STOP.
            print(f"🛑 Non-business rules compliant: {rule_message}")

            # Reset UI and Return (Don't run the engine)
            run_button.description = "Run Context Engine"
            run_button.disabled = False
            return

        # --- STEP 2: Run the Engine (Only if Valid) ---
        try:
            print(f"{rule_message}") # Print "Prompt Compliant" success message
            run_button.description = "Running..."

            execute_and_display(
                goal=user_goal,
                config=config,
                client=client,
                moderation_active=is_moderated
            )
        except NameError:
            print("🛑 Error: Engine dependencies not found. Please ensure previous notebook cells have been run.")
        except TypeError as e:
            print(f"🛑 Execution Error (Arguments Mismatch): {e}")
        except Exception as e:
            print(f"🛑 An unexpected error occurred: {e}")
        finally:
            # 3. Reset UI state regardless of success or failure
            run_button.description = "Run Context Engine"
            run_button.disabled = False

# 4. Bind Events
goal_dropdown.observe(on_dropdown_change, names='value')
run_button.on_click(on_run_click)

# 5. Render Interface
ui = widgets.VBox([
    widgets.HTML("<h3>🚀 Context Engine Control Deck</h3>"),
    widgets.HTML("<i>Select a goal to auto-fill, OR type directly in the box below.</i>"),
    goal_dropdown,
    goal_input,
    widgets.HBox([run_button], layout=widgets.Layout(justify_content='space-between', margin='10px 0 0 0')),
    widgets.HTML("""<hr style='border-top: 2px solid #ccc;'>"""),
    output_area
])

display(ui)